Puszczenie wielu eksprymentów

In [ ]:
import sys
import os
from pathlib import Path
import papermill as pm
import pandas as pd

sys.path.append(str(Path.cwd().parent))

In [ ]:
NOTEBOOK = Path.cwd() / "resnet_trening.ipynb"

configurations = pd.read_csv(
    "configurations.csv",
    sep=";"
).to_dict(orient="records")

for config in configurations:
    if pd.isna(config["weights_path"]):
        config["weights_path"] = None

print(f"Liczba konfiguracji: {len(configurations)}")
print(configurations)

In [ ]:
for i, config in enumerate(configurations, start=1):

    print(f"\nRozpoczynam eksperyment {i}/{len(configurations)}")

    output_path = "temporary.ipynb"

    try:
        pm.execute_notebook(
            NOTEBOOK,
            output_path,
            parameters=config
        )

    except pm.PapermillExecutionError as e:
        if "Ta kombinacja datasetu i hiperparametrów była już testowana." in str(e):
            print("Pominięto eksperyment.")
        else:
            raise

    finally:
        if os.path.exists(output_path):
            os.remove(output_path)

    print(f"Zakończono eksperyment {i}")

In [ ]:
NOTEBOOK = Path.cwd() / "wyniki.ipynb"
results_dir = Path.cwd() / "wyniki"
metrics_file = results_dir / "wyniki.csv"

processed_ids = (
    pd.read_csv(metrics_file)["Learning_ID"].tolist()
    if metrics_file.exists()
    else []
)

max_id = max(
    (
        int(path.stem)
        for path in results_dir.glob("*.pth")
        if path.stem.isdigit()
    ),
    default=0
)

configurations = [
    {"learning_id": i}
    for i in range(1, max_id + 1)
    if i not in processed_ids
]

print(f"Liczba konfiguracji: {len(configurations)}")
print(configurations)

In [ ]:
for i, config in enumerate(configurations, start=1):

    print(f"\nRozpoczynam eksperyment {i}/{len(configurations)}")

    output_path = "temporary.ipynb"

    try:
        pm.execute_notebook(
            NOTEBOOK,
            output_path,
            parameters=config
        )

    except pm.PapermillExecutionError as e:
        if "Ta kombinacja datasetu i hiperparametrów była już testowana." in str(e):
            print("Pominięto eksperyment.")
        else:
            raise

    finally:
        if os.path.exists(output_path):
            os.remove(output_path)

    print(f"Zakończono eksperyment {i}")